# 02_xai.ipynb

In [ ]:
# Mount colab
import os
import sys
from pathlib import Path

if True:
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

# Paths
PROJECT_ROOT: Path = Path(f"/content/drive/MyDrive/shared/python/projects/nlp_projects/text_classification/climate_skepticism")
os.chdir(PROJECT_ROOT)

# Add the PROJECT_ROOT directory to the Python path to import modules
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"ℹ️ Current Python version: {sys.version}")

ℹ️ $$ \ Current \ Python \ version: \ 3.12.13 \ (main, \ Mar \  \ 4 \ 2026, \ 09:23:07) \ [GCC \ 11.4.0]$$

In [ ]:
# Install reqs
# TODO: make logic to pick either requirements or snapshot, or something in-between
# to allow for flexibility during experimentation but also ensure reproducibility at the end
if True:
    %pip install -r "requirements/02_xai.txt"

In [ ]:
# -- project imports --
from src.config import load_configs
from src.utils.generic import set_seed
from src.utils.xai import build_explainer, save_shap_text_plot

import torch
from typing import Literal
import pandas as pd
import numpy as np
from pprint import pprint
from matplotlib import pyplot as plt
import seaborn as sns
from datasets import load_dataset, DatasetDict, ClassLabel
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig
)
from peft import PeftModel
from transformers import pipeline

import shap
from transformers_interpret import SequenceClassificationExplainer

# Set seaborn style
sns.set_style('darkgrid')

In [ ]:
# Initialize configs
config = load_configs()
set_seed(config.env.seed)

# Set constants
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAKE_BINARY: bool = False

print(f"ℹ️ Using device: {DEVICE}")

if MAKE_BINARY:
    FINETUNED_MODEL_NAME: str = config.model.finetuned_model_binary + "-" + config.model.version
    print(f"ℹ️ Making binary classification model: {FINETUNED_MODEL_NAME}")
else:
    FINETUNED_MODEL_NAME: str = config.model.finetuned_model_multiclass + "-" + config.model.version
    print(f"ℹ️ Making multiclass classification model: {FINETUNED_MODEL_NAME}")

pprint(config.to_dict())

---
## Data

[Frugal AI Challenge](https://huggingface.co/datasets/QuotaClimat/frugalaichallenge-text-train) dataset

In [ ]:
# Download the dataset
raw_datasets = load_dataset(config.data.dataset_repo)
print(raw_datasets)

In [ ]:
# Get number of labels for the model config
labels = sorted(raw_datasets["train"].unique("label"))
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

print(f"Number of labels: {num_labels}")
print(f"Label to ID mapping: {label2id}")
print(f"ID to Label mapping: {id2label}")

In [ ]:
# Rename text column
raw_datasets = raw_datasets.rename_column("quote", "text")

# Cast 'label' column to ClassLabel type for stratification
raw_datasets = raw_datasets.cast_column(
    "label",
    ClassLabel(num_classes=len(id2label), names=list(id2label.values()))
)

# Create validation split
valid_test_split = raw_datasets["test"].train_test_split(
    test_size=0.5, stratify_by_column="label",
    seed=config.env.seed
)
raw_datasets["valid"] = valid_test_split["train"]
raw_datasets["test"] = valid_test_split["test"]
print(raw_datasets)

In [8]:
# Define the filtering function
def filter_labels(example):
    labels_to_drop = ['7_fossil_fuels_needed']
    return example["label"] not in labels_to_drop

# Collapse labels -> binary
def collapse_labels(example):
    label_zero = '0_not_relevant'
    if example["label"] == label_zero:
        example["label"] = 0
    else:
        example["label"] = 1
    return example

# Processed dataset
if False:  # FIXME: replace constant bool with a condition/config
    raw_datasets = raw_datasets.filter(filter_labels)
if MAKE_BINARY:
    raw_datasets = raw_datasets.map(collapse_labels)

In [ ]:
# Check label proportions in sets
proportion_df = pd.DataFrame()

for key in raw_datasets.keys():
    proportion_df = pd.concat([
        proportion_df,
        raw_datasets[key]
            .to_pandas()['label']
            .value_counts(normalize=True)
            .to_frame()
            .rename(columns={'proportion': f'{key}_proportion'})
    ], axis=1)

sns.heatmap(proportion_df, annot=True, fmt=".2f")

## Model evaluation

### Load model

In [ ]:
adapter_checkpoint_path = f"Francesco-A/{FINETUNED_MODEL_NAME}"

auto_config = AutoConfig.from_pretrained(
    config.model.base_model,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

base_model = AutoModelForSequenceClassification.from_pretrained(
    config.model.base_model, config=auto_config
)

tokenizer = AutoTokenizer.from_pretrained(config.model.base_model)
model = PeftModel.from_pretrained(base_model, adapter_checkpoint_path).to(DEVICE)
model.eval()

### SHAP explainer

In [ ]:
# Build the explainer
explainer = build_explainer(model, tokenizer, DEVICE)

# Pick one text for each class
text_samples = raw_datasets["test"].to_pandas().groupby("label").apply(lambda x: x.sample(1, random_state=config.env.seed)).reset_index(drop=True)

# Explain the model's predictions
shap_values = explainer(text_samples["text"], fixed_context=1, batch_size=5)

In [ ]:
pipeline_classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=DEVICE)
pipeline_result = pipeline_classifier(text_samples['text'].tolist())
pprint(pipeline_result)

In [20]:
# Plot a sentence's explanation inline
#   - BUG(bad shap colors): https://stackoverflow.com/questions/79520326/shap-text-plot-does-not-show-properly-in-notebook/79836591#79836591
shap.plots.text(shap_values[3])

In [23]:
if True:
    save_shap_text_plot(shap_values, "evals/images/xai/one_sample_all_classes.html", sample_index=3)
    # save_shap_text_plot(shap_values, "evals/images/xai/all_samples_all_classes.html", sample_index='all')
    save_shap_text_plot(
        shap_values,
        "evals/images/xai/one_sample_one_class.html",
        sample_index=5,
        class_id=5,
    )

In [ ]:
if False:
    shap.plots.bar(shap_values.abs.mean(0))
    shap.plots.bar(shap_values.abs.sum(0))

### transformers-interpret

In [ ]:
cls_explainer = SequenceClassificationExplainer(
    model,
    tokenizer)
word_attributions = cls_explainer("Climate is not changing.")
# word_attributions = cls_explainer(text_samples['text'][3])

In [ ]:
print(cls_explainer.predicted_class_index)
print(cls_explainer.predicted_class_name)

In [21]:
cls_explainer.visualize("distilbert_viz.html")

True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
1,1_not_happening (1.00),1_not_happening,1.76,#s Climate is not changing . #/s


True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
1,1_not_happening (1.00),1_not_happening,1.76,#s Climate is not changing . #/s


## Requirements (full snapshot)

In [ ]:
!pip freeze > snapshots/02_xai.txt